# Data Loading And Dataset Profiling

This notebook exercises the source-loader layer. It proves that structured, semi-structured, and document-like enterprise inputs become `SourceRecord` objects with stable IDs and source-level evidence.

In [ ]:
from pathlib import Path
import sys, json, tempfile

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
from kg_construction.application.dataset_profiler import DatasetProfiler
from kg_construction.infrastructure.loaders import (
    APIResponseLoader, ChatTranscriptLoader, EmailLoader, ExcelLoader,
    MarkdownLoader, PDFLoader, ReportLoader, SQLDatabaseLoader, XMLLoader,
)
from kg_construction.infrastructure.loaders.csv_loader import CsvLoader
from kg_construction.infrastructure.loaders.json_loader import JsonLoader
from kg_construction.infrastructure.loaders.text_loader import TextLoader

fixtures = ROOT / 'tests' / 'fixtures' / 'enterprise_loaders'
loaders = {
    'csv': lambda: CsvLoader().load(ROOT / 'examples/data_to_kg/sample.csv'),
    'json': lambda: JsonLoader().load(ROOT / 'examples/data_to_kg/sample.json'),
    'text': lambda: TextLoader().load(ROOT / 'examples/data_to_kg/sample.txt'),
    'excel': lambda: ExcelLoader().load(fixtures / 'service_inventory.xlsx'),
    'sql': lambda: SQLDatabaseLoader().load(fixtures / 'enterprise.sqlite'),
    'api_response': lambda: APIResponseLoader().load(fixtures / 'api_response.json'),
    'xml': lambda: XMLLoader().load(fixtures / 'services.xml'),
    'markdown': lambda: MarkdownLoader().load(fixtures / 'incident_review.md'),
    'report': lambda: ReportLoader().load(fixtures / 'ops_report.report'),
    'pdf': lambda: PDFLoader().load(fixtures / 'incident_brief.pdf'),
    'email': lambda: EmailLoader().load(fixtures / 'incident_email.eml'),
    'chat': lambda: ChatTranscriptLoader().load(fixtures / 'support_chat.chat'),
}

profiler = DatasetProfiler()
summary = []
for name, load in loaders.items():
    sources = load()
    profile = profiler.profile_sources(sources, input_path=name)
    summary.append({
        'source': name,
        'source_records': len(sources),
        'rows': sum(len(source.records) for source in sources),
        'characters': sum(len(source.content or '') for source in sources),
        'profile_type': profile.data_type,
        'schema_quality': round(profile.schema_quality_score, 3),
        'source_evidence': all(bool(source.metadata.get('source_evidence')) for source in sources),
    })
print(json.dumps(summary, indent=2))

In [ ]:
# Inspect one loaded source record.
source = loaders['email']()[0]
print(source.source_id)
print(source.source_type)
print(source.metadata['headers'])
print(source.metadata['source_evidence'][0]['location'])